# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 4

# Task a : Unigrams, bigrams and trigrams in the corpus

An **n-gram** is a run of `n` tokens taken in order from the text: unigrams are
single words, bigrams are pairs of neighbouring words and trigrams are triples.

The corpus is `austen-emma.txt` from the Gutenberg corpus used in LAB 3. As the
question asks, the tokens are only lowercased and stripped of punctuation -
stopwords are **not** removed, so very common pairs like *of the* stay in the counts.

In [1]:
import sys
sys.path.append("..")

from nltk.corpus import gutenberg

from preprocessing import ensure_corpus, remove_punctuation

ensure_corpus("gutenberg")

words = [word.lower() for word in gutenberg.words("austen-emma.txt")]
tokens = remove_punctuation(words)

print("total tokens:", len(tokens))
print(tokens[:15])

total tokens: 161980
['emma', 'by', 'jane', 'austen', '1816', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich']


In [2]:
from nltk import ngrams, FreqDist

unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

for name, grams in [("unigrams", unigrams), ("bigrams", bigrams), ("trigrams", trigrams)]:
    print(name, "->", len(grams), "total,", len(set(grams)), "distinct")
    print(FreqDist(grams).most_common(10))
    print()

unigrams -> 161980 total, 7103 distinct
[(('to',), 5242), (('the',), 5204), (('and',), 4897), (('of',), 4293), (('i',), 3192), (('a',), 3130), (('it',), 2529), (('her',), 2490), (('was',), 2400), (('she',), 2364)]

bigrams -> 161979 total, 65922 distinct


[(('to', 'be'), 608), (('of', 'the'), 566), (('it', 'was'), 449), (('in', 'the'), 446), (('i', 'am'), 395), (('she', 'had'), 334), (('she', 'was'), 331), (('had', 'been'), 308), (('it', 'is'), 301), (('mr', 'knightley'), 299)]

trigrams -> 161978 total, 131040 distinct
[(('i', 'do', 'not'), 136), (('i', 'am', 'sure'), 109), (('she', 'could', 'not'), 73), (('a', 'great', 'deal'), 64), (('it', 'would', 'be'), 63), (('would', 'have', 'been'), 60), (('do', 'not', 'know'), 55), (('it', 'was', 'not'), 55), (('it', 'was', 'a'), 53), (('she', 'had', 'been'), 53)]



# Task b : Probability of a sentence using bigrams

The probability of a sentence is the chain rule of its words. A **bigram model**
approximates it by assuming each word depends only on the word just before it:

$$P(w_1 w_2 \dots w_n) \approx \prod_{i=1}^{n} P(w_i \mid w_{i-1})
\qquad \text{where} \qquad
P(w_i \mid w_{i-1}) = \frac{count(w_{i-1}, w_i)}{count(w_{i-1})}$$

`<s>` and `</s>` are added around every sentence so that the first and the last
word also have a bigram, which is what lets the model prefer words that actually
start or end a sentence.

In [3]:
train = ["This is a dog",
         "This is a cat",
         "I love my cat",
         "This is my name"]

sentences = [["<s>"] + sentence.lower().split() + ["</s>"] for sentence in train]

unigram_counts = FreqDist(word for sentence in sentences for word in sentence)
bigram_counts = FreqDist(bigram for sentence in sentences for bigram in ngrams(sentence, 2))

print("unigram counts:", dict(unigram_counts))
print()
print("bigram counts :", dict(bigram_counts))

unigram counts: {'<s>': 4, 'this': 3, 'is': 3, 'a': 2, 'dog': 1, '</s>': 4, 'cat': 2, 'i': 1, 'love': 1, 'my': 2, 'name': 1}

bigram counts : {('<s>', 'this'): 3, ('this', 'is'): 3, ('is', 'a'): 2, ('a', 'dog'): 1, ('dog', '</s>'): 1, ('a', 'cat'): 1, ('cat', '</s>'): 2, ('<s>', 'i'): 1, ('i', 'love'): 1, ('love', 'my'): 1, ('my', 'cat'): 1, ('is', 'my'): 1, ('my', 'name'): 1, ('name', '</s>'): 1}


In [4]:
test = ["<s>"] + "This is my cat".lower().split() + ["</s>"]

probability = 1.0
for w1, w2 in ngrams(test, 2):
    p = bigram_counts[(w1, w2)] / unigram_counts[w1]
    probability *= p
    print(f"P({w2} | {w1}) = {bigram_counts[(w1, w2)]}/{unigram_counts[w1]} = {p:.4f}")

print()
print("P('This is my cat') =", probability)

P(this | <s>) = 3/4 = 0.7500
P(is | this) = 3/3 = 1.0000
P(my | is) = 1/3 = 0.3333
P(cat | my) = 1/2 = 0.5000
P(</s> | cat) = 2/2 = 1.0000

P('This is my cat') = 0.125


# Task c : Predicting the next word with the bigram and trigram models

To guess the next word, look at the words just before it and pick the word that
followed them most often in the corpus.

- the **bigram** model looks at the last 1 word
- the **trigram** model looks at the last 2 words

`ConditionalFreqDist` does the counting: for every context it keeps a count of
the words that came after it. `.max()` gives the most common one and `.freq()`
gives its probability, that is, its share of all the words seen after that
context. The n-grams built in Task a are reused here.

In [5]:
from nltk import ConditionalFreqDist

bigram_model = ConditionalFreqDist(bigrams)
trigram_model = ConditionalFreqDist(((w1, w2), w3) for w1, w2, w3 in trigrams)

for word in ["mr", "she", "it", "a"]:
    next_word = bigram_model[word].max()
    print(word, "->", next_word, round(bigram_model[word].freq(next_word), 3))

mr -> knightley 0.259
she -> had 0.141
it -> was 0.178
a -> very 0.065


In [6]:
for pair in [("mr", "knightley"), ("she", "could"), ("it", "was"), ("a", "great")]:
    next_word = trigram_model[pair].max()
    print(" ".join(pair), "->", next_word, round(trigram_model[pair].freq(next_word), 3))

mr knightley -> s 0.1
she could -> not 0.424
it was -> not 0.122
a great -> deal 0.471


The trigram model guesses better, because two words are a stronger clue than one.
After `she` the bigram model can only offer `had` with a probability of 0.14, and
that word fits almost any sentence. After `she could` the trigram model gives
`not` with 0.42, and after `a great` it gives `deal` with 0.47.

The catch is that two-word contexts are much rarer. If a pair never appeared in
*Emma* the trigram model has nothing to say, while the bigram model nearly always
has an answer.

# Task d : Perplexity of a test text

**Perplexity** says how surprised a model is by a text it has not seen before. A
low number means the model expected the text, so a lower perplexity is better.

$$PP(W) = P(w_1 w_2 \dots w_N)^{-1/N}$$

Two practical points:

- the probabilities are added up as logs and turned back at the end, because
  multiplying thousands of tiny numbers would give 0
- an n-gram that never appeared in training would have probability 0, so 1 is
  added to every count (**add-one smoothing**) and the vocabulary size `V` to
  every context count

*Emma* is split 90:10, so the model is trained on the first part and tested on a
part it has never seen. The same function works for both models: `n=2` uses one
previous word as context and `n=3` uses two.

In [7]:
import math

split = int(0.9 * len(tokens))
train, test = tokens[:split], tokens[split:]
V = len(set(train))

def perplexity(words, n):
    ngram_counts = FreqDist(ngrams(train, n))
    context_counts = FreqDist(ngrams(train, n - 1))
    log_prob = 0
    for gram in ngrams(words, n):
        p = (ngram_counts[gram] + 1) / (context_counts[gram[:-1]] + V)
        log_prob += math.log2(p)
    return 2 ** (-log_prob / len(words))

In [8]:
print("bigram  perplexity:", round(perplexity(test, 2), 2))
print("trigram perplexity:", round(perplexity(test, 3), 2))

bigram  perplexity: 1248.85


trigram perplexity: 4729.64


The trigram model scores worse here, even though Task c showed it makes better
predictions. The reason is that two-word contexts are much rarer, so most trigrams
in the test part were never seen in training and get only the small smoothed
probability. The bigram model has seen more of its contexts before, so it is less
surprised overall.